# OpenCV Setup and Your First Pipeline

> **Beginner · Foundation**


## Why this matters

A reliable environment and a small, explicit pipeline let beginners focus on the image-processing idea rather than display and path errors.

**Where it appears:** Reading, transforming, displaying, and saving images or video frames in a reproducible workflow.


## Learning Objectives

- Verify an OpenCV installation and understand its module layout
- Understand the BGR-vs-RGB convention and why it matters immediately
- Build a first minimal, function-based image pipeline


## Prerequisites

01 Python Foundations and 02 NumPy for Images

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

`cv2.imread`, `cv2.imwrite`, `cv2.cvtColor`, image `shape`, `VideoCapture`

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### OpenCV Basics and Environment

OpenCV loads and stores color images in **BGR** channel order, not RGB --
a historical artifact from early Windows camera APIs. Every other Python
imaging/plotting library (Matplotlib, PIL, most ML frameworks) expects
RGB. Forgetting this conversion is the single most common OpenCV beginner
bug (colors look swapped -- skin looks blue, sky looks orange).


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


### 1. Confirming the installation

Check the OpenCV build and which optional modules (e.g. `cv2.face`, `cv2.dnn`) are available, since several later notebooks depend on `opencv-contrib-python`.


In [ ]:
import cv2


def opencv_report() -> dict:
    return {
        "version": cv2.__version__,
        "has_contrib_face_module": hasattr(cv2, "face"),
        "has_dnn_module": hasattr(cv2, "dnn"),
        "build_info_snippet": cv2.getBuildInformation().splitlines()[0],
    }


for k, v in opencv_report().items():
    print(f"{k}: {v}")

### 2. BGR vs RGB, made concrete

Let's load a real image using OpenCV. OpenCV reads images in BGR format, while Matplotlib displays them in RGB format. If we forget to convert, the Red and Blue channels are swapped! This is the most common beginner mistake in OpenCV, famously causing 'Smurf skin' (blue faces).


In [ ]:
import numpy as np
from cv_utils import show_grid

# Load a real human face image (Lena is standard in CV, but we use a modern portrait)
bgr_image = load_real_image("images/faces", "face.jpg")

# The CORRECT way: convert BGR to RGB before handing off to Matplotlib
correctly_shown = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)

show_grid(
    [
        (
            "Raw array (WRONG: Blue skin)",
            bgr_image,
        ),  # Matplotlib will assume this BGR array is RGB
        ("cv2.cvtColor (CORRECT)", correctly_shown),
    ],
    cols=2,
    figsize_scale=4,
)

### 3. A minimal function-based pipeline

Every later notebook follows this same shape: a small pipeline function that takes an image in and returns a result, instead of flat inline steps.


In [ ]:
def minimal_pipeline(image: np.ndarray) -> np.ndarray:
    """Grayscale -> blur -> edges. The 'hello world' OpenCV pipeline, as one function."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    # Blur is critical before edge detection to remove high-frequency noise!
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, 50, 150)
    return edges


# Let's run this pipeline on a real image containing edges (coins)
scene = load_real_image("images/objects", "coins.jpg")
result = minimal_pipeline(scene)

show_grid([("Input", scene), ("Edge Detection Output", result)])

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — OpenCV Basics and Environment: Processing a Real Video Stream

In production, computer vision pipelines process video streams. A video is just a sequence of images (frames). Here, we use `cv2.VideoCapture` to open a real video file, run our edge detection filter on the first 30 frames, and measure the Frames Per Second (FPS).


In [ ]:
import time


def process_video_stream(video_path, process_fn, max_frames=30):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"Could not open video {video_path}")

    total_time = 0.0
    processed_frames = []

    for _ in range(max_frames):
        ret, frame = cap.read()
        if not ret:
            break  # End of video

        t0 = time.perf_counter()
        processed = process_fn(frame)
        t1 = time.perf_counter()

        total_time += t1 - t0
        processed_frames.append(processed)

    cap.release()

    avg_fps = len(processed_frames) / total_time
    print(
        f"Processed {len(processed_frames)} frames in {total_time * 1000:.2f} ms (Average FPS: {avg_fps:.2f})"
    )
    return processed_frames


# Define standard image filter process
def my_filter(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 1.5)
    return cv2.Canny(blurred, 50, 150)


# Run on our real traffic video
video_file = get_real_data("video", "traffic.mp4")
output = process_video_stream(video_file, my_filter, max_frames=30)

print("Video pipeline complete! Let's view the last processed frame:")
show(output[-1], "Last Frame (Edge Detection)")

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — OpenCV Basics and Environment
1. Write `bgr_to_rgb_safe(image)` that also handles grayscale input (returns it unchanged).
2. Extend `opencv_report()` to also report whether CUDA support was compiled in.
3. Modify `minimal_pipeline` to accept the blur kernel size and Canny thresholds as parameters.

Use the empty cell below to work through them.


#### Solutions — OpenCV Basics and Environment

In [ ]:
# Solution 1: bgr_to_rgb_safe that handles grayscale input
def bgr_to_rgb_safe(image: np.ndarray) -> np.ndarray:
    """Convert BGR image to RGB; return unchanged if image is grayscale."""
    if image.ndim == 2 or image.shape[2] == 1:
        return image
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

In [ ]:
# Solution 2: opencv_report reporting CUDA support
def opencv_report() -> None:
    """Generate a diagnostic report of the OpenCV installation, including CUDA."""
    print("OpenCV Version:", cv2.__version__)
    try:
        cuda_devices = cv2.cuda.getCudaEnabledDeviceCount()
        print("CUDA Support: AVAILABLE")
        print(f"CUDA Enabled Devices: {cuda_devices}")
    except AttributeError:
        print("CUDA Support: NOT AVAILABLE (CPU-only build)")

In [ ]:
# Solution 3: minimal_pipeline accepting thresholds as parameters
def minimal_pipeline(
    image: np.ndarray, ksize: int = 5, low_thresh: int = 50, high_thresh: int = 150
) -> np.ndarray:
    """Run a gray -> blur -> Canny pipeline with parameterized settings."""
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (ksize, ksize), 0)
    return cv2.Canny(blurred, low_thresh, high_thresh)


opencv_report()

## Summary

You can verify OpenCV, respect BGR versus RGB, and write a minimal function-based image pipeline.

- **Best Practices:** Read images defensively, convert BGR only at display boundaries, and keep input, processing, and output steps separate.
- **Common Pitfalls:** Using `cv2.imshow` in a notebook, treating a failed read as an image, and mixing RGB and BGR arrays.